# Creating Kaggle submissions for CIFAR10 Classification (Parts 2, 3, 4, 5)

In [ ]:
from typing import Callable
from pathlib import Path

import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np


class CIFAR10TestOnly(torchvision.datasets.CIFAR10):
    def __init__(
        self,
        root: str | Path,
        download: bool,
        transform: Callable[[torch.Tensor], torch.Tensor],
    ) -> None:
        super().__init__(root=root, train=False, download=download, transform=transform)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int):
        image, class_label = super().__getitem__(index)
        return image, class_label


transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

batch_size = 128

test_dataset = CIFAR10TestOnly(root="./data", download=True, transform=transform_test)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
@torch.inference_mode()
def run_classification_test(model, test_dataloader):
    all_output = []
    for images, _ in test_dataloader:
        images = images.to(device)
        # Calculate outputs by running images through the model
        outputs = model(images)
        # The class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        all_output.append(predicted.cpu().numpy())
    return np.concatenate(all_output)

## Instantiate your model here:

In [ ]:
model = <your model>
model = model.to(device)
model.eval()

## Load the model weights:

In [ ]:
model.load_state_dict(torch.load(<path to your saved model>))

## Submit the generated .csv file to Kaggle:

In [ ]:
import pandas as pd

predictions = run_classification_test(model, test_dataloader)
ids = np.arange(len(predictions))
df = pd.DataFrame({"id": ids, "prediction": predictions})
df.to_csv("./classification_test_predictions.csv", index=False)